# Text Classification: Optimization & Multiclass Extensions

In this notebook, we dive deep into the mechanics of classical machine learning.

### Objectives
1.  **Custom Implementation**: Use our scratch-built ElasticNet Logistic Regression with L1 + L2 Regularization.
2.  **Hyperparameter Analysis**: Visualize how parameters of Regularization $C$, Kernel $\gamma$, Polynomial Degree affect SVM performance using Heatmaps.
3.  **Model Comparison**: Compare our custom Logit against Sklearn's SVM on a Binary Text Classification task.
4.  **Multiclass Extension**: Extend our binary custom model to handle 4 classes using a One-vs-One strategy.

### Dataset
We use the **AG News** dataset with 4 topics: World, Sports, Business, Sci/Tech.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.multiclass import OneVsOneClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_curve, auc

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from elastic_logit import ElasticNetLogit

sns.set_theme(style="whitegrid")

## 1. ElasticNet Logic Implementation

We'll use the custom class `ElasticNetLogit` which implements:

$$ L(w) = -\sum [y \log \sigma + (1-y) \log(1-\sigma)] + \gamma ||w||_1 + \beta ||w||_2^2 $$

See `src/elastic_logit.py` for the gradient derivation and code.

In [ ]:
# 1. Sanity Check: Synthetic Data
X_synth, y_synth = make_classification(n_samples=200, n_features=2, n_redundant=0,
                                       n_informative=2, random_state=42, n_clusters_per_class=1)

model = ElasticNetLogit(beta=0.1, gamma=0.1, learning_rate=0.01, max_iter=500)
model.fit(X_synth, y_synth)

plt.figure(figsize=(8, 4))
plt.plot(model.loss_history)
plt.title("Training Loss (Synthetic Data)")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

print(f"Final Weights: {model.w}")

## 2. Data Preparation

We perform text preprocessing and TF-IDF vectorization.
For the first part, we select a binary subset: Class 1, World, vs Class 4, Sci/Tech.

In [ ]:
try:
    data = pd.read_csv("../../data/ag_news_train.csv", header=None)
    data.columns = ["Class Index", "Title", "Description"]
except FileNotFoundError:
    print("Downloading dataset...")
    !wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv -O temp.csv
    data = pd.read_csv("temp.csv", header=None)
    data.columns = ["Class Index", "Title", "Description"]

data["text"] = data["Title"] + " " + data["Description"]
data["text"] = data["text"].str.lower().str.replace(r'[^\w\s]', '', regex=True)

binary_df = data[data["Class Index"].isin([1, 4])].sample(4000, random_state=42).copy()
binary_df["label"] = (binary_df["Class Index"] == 4).astype(int)

vectorizer = TfidfVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(binary_df["text"]).toarray()
y = binary_df["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print(f"Class Balance: {np.bincount(y_train)}")

## 3. Tuning Custom ElasticNet

We optimize our `ElasticNetLogit` (defined in `src/elastic_logit.py`) by tuning:
*   `alpha` (Learning Rate)
*   `beta` (L2 Ridge Penalty)
*   `gamma` (L1 Lasso Penalty)

In [ ]:
betas = [0, 0.01, 0.1]
gammas = [0, 0.01, 0.1]
best_acc = 0
best_params = {}
heatmap_data = np.zeros((len(betas), len(gammas)))

print("Tuning ElasticNet...")
for i, b in enumerate(betas):
    for j, g in enumerate(gammas):
        model = ElasticNetLogit(beta=b, gamma=g, learning_rate=0.1, max_iter=200)
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        heatmap_data[i, j] = acc

        if acc > best_acc:
            best_acc = acc
            best_params = {'beta': b, 'gamma': g}

print(f"Best ElasticNet Accuracy: {best_ac} with params {best_params}")

plt.figure(figsize=(6, 5))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", xticklabels=gammas, yticklabels=betas, cmap="viridis")
plt.xlabel("Gamma (L1)")
plt.ylabel("Beta (L2)")
plt.title("ElasticNet Accuracy Heatmap")
plt.show()

## 4. Tuning SVM with RBF Kernel

We examine how the SVM RBF kernel behaves.
*   **C, Regularization**: Inverse of regularization strength.
*   **Gamma, Kernel Coefficient**: Defines how far the influence of a single training example reaches.

**Hypothesis**: Low Gamma = underfitting; High Gamma = overfitting.

In [ ]:
param_grid = {
    'C': np.logspace(-1, 2, 5),
    'gamma': np.logspace(-2, 1, 5)
}

svm = SVC(kernel='rbf', probability=True)
grid = GridSearchCV(svm, param_grid, cv=3, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

scores = grid.cv_results_['mean_test_score'].reshape(len(param_grid['C']), len(param_grid['gamma']))

plt.figure(figsize=(8, 6))
sns.heatmap(scores, annot=True, cmap="magma", fmt=".3f",
            xticklabels=np.round(param_grid['gamma'], 3),
            yticklabels=np.round(param_grid['C'], 3))
plt.xlabel("Gamma")
plt.ylabel("C (Regularization)")
plt.title("SVM RBF F1-Score Landscape")
plt.show()

best_svm = grid.best_estimator_
print(f"Best SVM Params: {grid.best_params_}")

## 4. Model Comparison: ROC Curves

We compare the best ElasticNet with Linear decision boundary vs. the best SVM RBF with Non-linear decision boundary.

In [ ]:
def get_proba_custom(model, X):
    X_aug = np.hstack((np.ones((X.shape[0], 1)), X))
    z = X_aug @ model.w
    return 1 / (1 + np.exp(-np.clip(z, -20, 20)))

logit_final = ElasticNetLogit(**best_params, learning_rate=0.1, max_iter=500)
logit_final.fit(X_train, y_train)

y_prob_logit = get_proba_custom(logit_final, X_test)
y_prob_svm = best_svm.predict_proba(X_test)[:, 1]

fpr_l, tpr_l, _ = roc_curve(y_test, y_prob_logit)
auc_l = auc(fpr_l, tpr_l)

fpr_s, tpr_s, _ = roc_curve(y_test, y_prob_svm)
auc_s = auc(fpr_s, tpr_s)

plt.figure(figsize=(8, 6))
plt.plot(fpr_l, tpr_l, label=f'ElasticNet (AUC = {auc_l:.3f})')
plt.plot(fpr_s, tpr_s, label=f'SVM RBF (AUC = {auc_s:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

## 5. Multiclass Extension

Our `ElasticNetLogit` is a binary classifier. To handle the full AG News dataset (4 classes), we wrap it in Sklearn's `OneVsOneClassifier`.

This trains $\frac{N(N-1)}{2} = 6$ binary classifiers (World vs Sports, World vs Biz, etc.) and votes for the final class. This demonstrates that our custom scratch-built class is Sklearn-compatible.

In [ ]:
multi_df = data.sample(3000, random_state=42).copy()
X_multi = vectorizer.fit_transform(multi_df["text"]).toarray()
y_multi = multi_df["Class Index"].values

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(X_multi, y_multi, test_size=0.3, random_state=42)
ovo_classifier = OneVsOneClassifier(ElasticNetLogit(beta=0.01, gamma=0.01, learning_rate=0.1, max_iter=200))

print("Training One-vs-One Multiclass Classifier...")
ovo_classifier.fit(X_m_train, y_m_train)

preds_multi = ovo_classifier.predict(X_m_test)
acc_multi = accuracy_score(y_m_test, preds_multi)
print(f"Multiclass Accuracy (Custom Logit + OvO): {acc_multi:.4f}")

cm = confusion_matrix(y_m_test, preds_multi)
labels = ["World", "Sports", "Business", "Sci/Tech"]

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix: Custom Multiclass Logit")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()

## Conclusion

1.  **Optimization**: The heatmaps reveal that SVMs are highly sensitive to the `Gamma` parameter. A narrow `Gamma` leads to underfitting, because the model is too smooth, while a high `Gamma` overfits, because the model creates islands around specific data points.
2.  **Linear vs Non-Linear**: The RBF SVM, being Non-Linear, slightly outperforms ElasticNet, which is Linear, indicating the text boundaries are not perfectly linear. However, the custom Logit is competitive and much faster to train.
3.  **Extensibility**: We successfully adapted a binary classifier built from scratch into a multiclass system using the One-vs-One strategy, achieving respectable accuracy on the full dataset.